In [2]:
"""
This file implements BitFit, which is a parameter efficient fine-tuning method.
It only updates the biases and not the weights.
"""

# from scripts.task3_data_preparation import prepare_data_qu
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import datasets
import sys
import os
import evaluate
import numpy as np
import torch

# Get the current working directory (which should be inside the 'notebooks' directory)
current_dir = os.getcwd()

# Go up one level to the parent directory
parent_dir = os.path.dirname(current_dir)

# Add the 'scripts' directory to the Python path
scripts_path = os.path.join(parent_dir, 'scripts')
sys.path.insert(0, scripts_path)


In [3]:
def compute_metrics(eval_pred, metric):
    """
    This function will be used for model evaluation. 

    args:
        eval_pred: the output of the model. Made up of logits and labels, where logits
        are unnormalized scores that the model predicts. These logits are typically 
        transformed into probabilities using a softmax function. 

    returns:
        - Dictionary: the keys represent different sub-datasets and the evaluation on them separately.
    """

    # In evaluation mode, the output of an autoregressive model like XGLM consists of logits 
    # corresponding to the score for each possible next token in the vocabulary given the previous sequence of tokens. 
    # The label is the ground truth—the actual token that comes next in the sequence according to the dataset, 
    # regardless of the model's prediction. It's the target that the model is trying to predict.
    logits, labels = eval_pred

    # Convert logits to probabilities for all possible next words
    probabilities = torch.softmax(logits, axis=-1)
    
    # You need the probabilities of the actual next tokens, referenced by the labels
    true_next_token_probabilities = np.take_along_axis(probabilities, np.expand_dims(labels, -1), axis=-1)

    # Perplexity is calculated using the probabilities of the true next tokens
    perplexity = metric.compute(predictions=true_next_token_probabilities.squeeze(), references=labels)
    
    return perplexity

In [4]:
# Steps
# 1) Prepare training data
# 2) pass in a batch of data to the model
# 3) compute the loss
# 4) update the bias parameters based on gradient 

In [5]:
# Load the dataset
dataset_path = "hackathon-pln-es/spanish-to-quechua"

data_files = {
"train": "data/train-00000-of-00001.parquet",
"validation": "data/validation-00000-of-00001.parquet",
"test": "data/test-00000-of-00001.parquet"
}
# load the dataset for training
dataset = datasets.load_dataset(dataset_path, data_files=data_files)
training_data = dataset["train"]

# prepare the dataloader
training_data_loader =  DataLoader(training_data, batch_size=6, shuffle=True)


In [13]:
# define the model
MODEL_NAME = "facebook/xglm-564M"

# load pre-trained model from the huggingface hub
xglm_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
# set to train mode
xglm_model.train()

# load a pre-trained tokenizer from the huggingface hub
xglm_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# since we are using a decoder only architecture, we need to perform the padding
# on the left side so that the model does not get confused while guessing the next token.
xglm_tokenizer.padding_side = 'left'


In [14]:
# define the optimizer
# Optimizers specified in the torch.optim package
optimizer = torch.optim.SGD(xglm_model.parameters(), lr=0.001, momentum=0.9)

# Setup evaluation, which will be used to calculate the loss.
metric = evaluate.load("perplexity", module_type="metric")

# define the loss function
loss_fn = torch.nn.CrossEntropyLoss()

# remember to freeze the weights and not the biases

In [ ]:
for i, data in enumerate(training_data_loader):
    # Zero your gradients for every batch!
    optimizer.zero_grad()
    
    # only get the "qu" sentences
    qu_sentences = data["qu"]

    # tokenize the sentences tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")
    tokenized = xglm_tokenizer(qu_sentences, padding=True, truncation=True, return_tensors="pt")
    input_ids = tokenized["input_ids"]
    attention_mask = tokenized["attention_mask"]
    print()

    # Make predictions for this batch
    outputs = xglm_model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs["logits"]
    print(outputs["labels"])
    break

    logits_flat = logits.view(-1, logits.size(-1))
    labels_flat = labels.view(-1)

    # Compute the loss
    loss = loss_fn(logits_flat, labels_flat)

    # Backward pass
    loss.backward()

    # Update the parameters
    optimizer.step()
    
